[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mist-medical/MIST/blob/main/examples/mist_heart_demo.ipynb)

# MIST End-to-End Demo — Heart Segmentation (MSD Task 2)

This notebook demonstrates the full [MIST](https://github.com/mist-medical/MIST) segmentation pipeline on the
[Medical Segmentation Decathlon](http://medicaldecathlon.com/) Task 2 (Heart / left atrium) dataset.

We will walk through:
1. Installing MIST and downloading the data
2. Converting the MSD dataset to MIST format with `mist_convert_msd`
3. Exploring the converted dataset
4. Running the pipeline stage by stage: analyze → tune config → preprocess → train
5. Reviewing the cross-validation evaluation metrics
6. Running inference on test images with `mist_predict`
7. Applying postprocessing with `mist_postprocess`
8. Comparing and ranking results with `mist_rank`
9. Visualizing the results

> **Requirements:** A GPU runtime is required. The free **T4 GPU** in Colab is
> enough to run the whole notebook (`Runtime → Change runtime type → T4 GPU`).
> If you have Colab Pro, selecting an **A100** will train and infer noticeably
> faster.

## Section 1: Setup

Install MIST with the training dependencies (includes NVIDIA DALI, PyTorch, and MONAI).

In [ ]:
!pip install "mist-medical[train]" -q

Download and extract the MSD Task 2 (Heart) dataset from Google Drive.
The dataset contains 20 MRI volumes for training and 10 for testing.

In [ ]:
# @title
import gdown

gdown.download(
    "https://drive.google.com/uc?id=1wEB2I6S6tQBVEPxir8cA5kFB8gTQadYY",
    output="Task02_Heart.tar",
    quiet=False,
)
!tar -xf Task02_Heart.tar

## Section 2: Convert to MIST Format

`mist_convert_msd` reads the MSD dataset's `dataset.json` and reorganizes the
images and labels into the MIST directory layout under `heart_mist/`, writing a
MIST `dataset.json` that describes the dataset for the downstream pipeline
stages. We inspect that file in the next section.

In [ ]:
!mist_convert_msd \
    --source Task02_Heart \
    --output heart_mist \
    --num-workers-conversion 2

## Section 3: Explore the converted dataset

The MIST `dataset.json` is a compact description of the segmentation task that
drives the rest of the pipeline. Let's print it and see what `mist_convert_msd`
filled in.

In [ ]:
import json

with open("heart_mist/dataset.json") as f:
    dataset = json.load(f)

dataset

The key fields are:

- **`task`** / **`modality`** — a human-readable name and the imaging modality
  (`mr` here).
- **`train-data`** / **`test-data`** — directories (relative to the JSON) holding
  one subfolder per patient.
- **`images`** — the named input channels mapped to filename patterns. This
  dataset has a single `MRI` channel; a multi-modal task (e.g. BraTS) would list
  several.
- **`mask`** — filename pattern(s) for the ground-truth label map.
- **`labels`** — the integer values that appear in the masks: `0` is background
  and `1` is the left atrium.
- **`final_classes`** — how those labels are grouped into the classes MIST
  evaluates. Here label `1` becomes the `left_atrium` class — the same class
  name you'll see in the metrics later.

Now let's load one training case and plot an axial slice with its ground-truth
left-atrium mask overlaid, so we know what the model is learning to segment.

In [ ]:
from pathlib import Path

import ants
import matplotlib.pyplot as plt
import numpy as np

train_dir = Path("heart_mist/raw/train")
example_id = sorted(p.name for p in train_dir.iterdir() if p.is_dir())[0]

mri  = ants.image_read(str(train_dir / example_id / "MRI.nii.gz")).numpy()
mask = ants.image_read(str(train_dir / example_id / "mask.nii.gz")).numpy()

# Pick the axial slice containing the most left-atrium voxels.
z = int(np.argmax(mask.sum(axis=(0, 1))))

fig, axes = plt.subplots(1, 2, figsize=(9, 5))
fig.suptitle(f"Training case: {example_id}  (axial slice {z})", fontsize=12)

axes[0].imshow(mri[:, :, z].T, cmap="gray", origin="lower")
axes[0].set_title("MRI")
axes[0].axis("off")

axes[1].imshow(mri[:, :, z].T, cmap="gray", origin="lower")
overlay = np.ma.masked_where(mask[:, :, z] == 0, mask[:, :, z])
axes[1].imshow(overlay.T, cmap="autumn", alpha=0.5, origin="lower")
axes[1].set_title("MRI + left-atrium mask")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Section 4: Run the pipeline — analyze, tune, preprocess, and train

MIST can run the whole pipeline with a single `mist_run_all` command. Here we run
the stages separately, for two reasons: it lets us **inspect and tune**
`config.json` — the single source of truth the analyzer writes and every later
stage reads — before committing to a training run, and it makes each step of the
pipeline explicit.

First, **`mist_analyze`** inspects the dataset (spacing, intensity, shapes) and
writes `heart_results/config.json` with defaults for spacing, patch size,
normalization, and the model, training, and inference settings.

In [ ]:
!mist_analyze \
    --data heart_mist/dataset.json \
    --results heart_results

### Configure the run in `config.json`

`config.json` is the single source of truth for the whole run, and it's just
JSON — so instead of passing a long list of flags to `mist_train`, we set
everything here before training. Two groups of settings:

**Hardware and speed**

- **`inference.tta.enabled` → `false`.** Test-time augmentation averages eight
  flipped passes per case — an ~8× inference cost for a marginal accuracy gain.
- **`inference.inferer.params.patch_overlap` → `0.25`** (from `0.5`) — fewer
  sliding-window steps per volume, for faster inference.

We leave `training.amp` alone: MIST automatically detects whether the GPU has
native BF16 support (Ampere or newer) and falls back to FP32 otherwise. On the
free Colab T4 (pre-Ampere) it disables BF16 for you — you'll see `AMP: off (FP32)`
in the training summary — so mixed precision never silently runs the slow
emulated path. (On an A100, BF16 stays on and is genuinely faster.)

**Model and training**

- **`model.architecture` → `nnunet-pocket`** — a lightweight nnU-Net variant
  (~700K parameters) that matches the full nnU-Net on a small, single-structure
  dataset while training much faster. MIST ships many architectures — try
  `nnunet`, `mednext-base`, or `swinunetr-base` for full-capacity models.
- **`training.folds` → `[0, 1]`** — MIST runs full 5-fold cross-validation and
  ensembles all five fold models by default; we train just two folds here.
- **`training.epochs` → `5`** — five epochs per fold (raise this substantially for
  real training).
- **`training.warmup_epochs` → `0`** — skip the linear LR warmup for this short run.
- **`spatial_config.patch_size` → `[64, 64, 48]`** — MIST would normally select
  128 × 128 × 96; a smaller patch trains faster. Delete this line to keep the
  analyzer's choice.

In [ ]:
import json

config_path = "heart_results/config.json"
with open(config_path) as f:
    config = json.load(f)

# Hardware and speed. (training.amp is left alone — MIST auto-detects BF16 support.)
config["inference"]["tta"]["enabled"] = False                        # skip 8x flip augmentation
config["inference"]["inferer"]["params"]["patch_overlap"] = 0.25     # fewer sliding windows

# Model and training.
config["model"]["architecture"] = "nnunet-pocket"
config["training"]["folds"] = [0, 1]
config["training"]["epochs"] = 5
config["training"]["warmup_epochs"] = 0
config["spatial_config"]["patch_size"] = [64, 64, 48]

with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print("architecture :", config["model"]["architecture"])
print("folds        :", config["training"]["folds"],
      "| epochs:", config["training"]["epochs"],
      "| patch_size:", config["spatial_config"]["patch_size"])
print("tta          :", config["inference"]["tta"]["enabled"],
      "| patch_overlap:", config["inference"]["inferer"]["params"]["patch_overlap"])

### Preprocess and train

With the run configured, the remaining commands are minimal. **`mist_preprocess`**
reorients, crops, resamples, and normalizes the images into NumPy arrays under
`heart_numpy/`, following the recipe in `config.json`. Then **`mist_train`** reads
every model and training choice from `config.json`, runs the two-fold
cross-validation, and — when it finishes — evaluates each fold's held-out
predictions and generates predictions for the test set.

In [ ]:
!mist_preprocess \
    --results heart_results \
    --numpy heart_numpy

In [ ]:
!mist_train \
    --results heart_results \
    --numpy heart_numpy

## Section 5: Review the cross-validation evaluation metrics

`mist_train` doesn't just train. After the folds finish, it runs each trained
fold's model on its held-out cases, writes those out-of-sample predictions to
`heart_results/predictions/train/raw/`, and evaluates them against the ground
truth. The per-patient scores and summary statistics are saved to
`heart_results/results.csv`.

By default MIST reports two metrics per class: **Dice** (volumetric overlap,
higher is better) and **HD95** (the 95th-percentile Hausdorff distance in mm,
lower is better). MIST can also compute average surface distance, surface Dice,
and BraTS-style lesion-wise metrics — see the
[evaluation documentation](https://mist-medical.readthedocs.io/en/latest/usage/#evaluation)
for the full list and how to enable them.

Let's load `results.csv` and print the summary statistics for the two default
metrics.

In [ ]:
import pandas as pd

results = pd.read_csv("heart_results/results.csv")

# Summary rows MIST appends below the per-patient scores.
summary_ids = ["Mean", "Std", "25th Percentile", "Median", "75th Percentile"]
metric_cols = [c for c in results.columns if c.endswith(("_dice", "_haus95"))]

results[results["id"].isin(summary_ids)].set_index("id")[metric_cols].loc[summary_ids]

## Section 6: Predict with `mist_predict`

`mist_predict` is MIST's standalone inference command — the one you'd reach for
to segment new images with an already-trained model. It runs sliding-window
inference with the trained fold models (two in this quick demo) and ensembles
their predictions. It expects a **paths CSV** with an `id` column (a unique
patient identifier) and one column per image modality, where each column holds
the path to that patient's image:

| id | MRI |
|----|-----|
| la_001 | /path/to/la_001/MRI.nii.gz |
| la_002 | /path/to/la_002/MRI.nii.gz |

Because our `dataset.json` defined a `test-data` directory, `mist_train` already
ran inference on the full test set (into `heart_results/predictions/test/`), and
`mist_analyze` already wrote the paths CSV to `heart_results/test_paths.csv`. To
demonstrate the standalone command without repeating that whole run, we copy just
the first two rows into a small CSV and predict on those.

In [ ]:
import pandas as pd

# MIST auto-generated this CSV during analysis because dataset.json set test-data.
test_paths = pd.read_csv("heart_results/test_paths.csv")

# Take just two cases so the standalone demo runs quickly.
demo_paths = test_paths.head(2)
demo_paths.to_csv("demo_paths.csv", index=False)
demo_paths

In [ ]:
!mist_predict \
    --models-dir heart_results/models \
    --config heart_results/config.json \
    --paths-csv demo_paths.csv \
    --output heart_predictions

`mist_predict` writes one segmentation per input case to `heart_predictions/`
(`<id>.nii.gz`) — here, just the two demo cases. The full test set was already
segmented by `mist_train` into `heart_results/predictions/test/`.

We don't score these because the MSD test set has no public labels; the
postprocessing and comparison sections below instead use the cross-validation
predictions, which do have ground truth.

## Section 7: Postprocess with `mist_postprocess`

The left atrium is a single connected structure, so we apply
`get_top_k_connected_components` to keep only the largest connected component
and drop any spurious detections outside the heart.

We run this on the **cross-validation predictions** in
`heart_results/predictions/train/raw/` rather than the test predictions, because
the training cases have ground-truth masks. (The MSD test set ships without
labels, so it can't be scored.)

By also passing `--paths-csv` (patient IDs and ground-truth mask paths) and
`--eval-config`, `mist_postprocess` evaluates the postprocessed masks in the same
run and writes the scores to `heart_postprocessed/postprocess_results.csv` — no
separate `mist_evaluate` call needed. We reuse the `evaluation_paths.csv` that
training already generated for the `id` and `mask` columns.

In [ ]:
import json

strategy = [
    {
        "transform": "get_top_k_connected_components",
        "apply_to_labels": [1],
        "per_label": False,
        "kwargs": {"top_k_connected_components": 1},
    }
]
with open("heart_strategy.json", "w") as f:
    json.dump(strategy, f, indent=2)

print("Postprocessing strategy saved to heart_strategy.json")

In [ ]:
!mist_postprocess \
    --base-predictions heart_results/predictions/train/raw \
    --output heart_postprocessed \
    --postprocess-strategy heart_strategy.json \
    --paths-csv heart_results/evaluation_paths.csv \
    --eval-config heart_results/config.json

## Section 8: Compare raw vs. postprocessed metrics

`mist_postprocess` already evaluated the postprocessed predictions and saved the
scores to `heart_postprocessed/postprocess_results.csv`. We line those up against
the baseline (raw prediction) metrics from Section 5 to see whether keeping only
the largest connected component helped.

In [ ]:
baseline = pd.read_csv("heart_results/results.csv").set_index("id")
postproc = pd.read_csv("heart_postprocessed/postprocess_results.csv").set_index("id")

comparison = pd.DataFrame({
    "baseline (mean)": baseline.loc["Mean", metric_cols],
    "postprocessed (mean)": postproc.loc["Mean", metric_cols],
})
comparison["change"] = comparison["postprocessed (mean)"] - comparison["baseline (mean)"]
comparison

### Ranking results with `mist_rank`

The table above is a quick manual comparison of two result sets. For a
principled comparison — especially across more than two candidates — MIST ships
`mist_rank`, which ranks N evaluation CSVs BraTS-style: for every
(patient, metric) cell it ranks the candidates from best (1) to worst (with
average tie handling), then aggregates to a mean rank per candidate. A **lower
average rank is better**, and because the ranking is per-patient it accounts for
case-by-case wins rather than just comparing dataset-wide means.

Every MIST evaluation writes a `results.csv` in the same format, so `mist_rank`
isn't limited to postprocessing strategies — you can rank any set of runs
evaluated on the same cases and metrics: different **models** (`--model`),
**loss functions** (`--loss`), **training configurations**, or **ensembles**.
Just point `--results` at each run's `results.csv` and give each a friendly
`--name`. Here we rank the raw predictions against the postprocessed ones.

Two optional outputs are worth knowing about: `--output-detailed-csv` writes a
per-metric breakdown of the mean ranks, and `--significance-csv` writes a
pairwise Wilcoxon signed-rank p-value matrix to test whether one candidate is
significantly better than another.

In [ ]:
!mist_rank \
    --results heart_results/results.csv heart_postprocessed/postprocess_results.csv \
    --names raw postprocessed \
    --output-csv heart_ranking.csv \
    --output-detailed-csv heart_ranking_detailed.csv

In [ ]:
# 'average_rank' aggregates the per-patient, per-metric ranks (lower is better).
pd.read_csv("heart_ranking.csv")

## Section 9: Visualize Results

We plot three axial slices for one cross-validation case, comparing the raw
prediction (top row) against the postprocessed prediction (bottom row). The model
prediction is shown as a red overlay and the ground-truth left-atrium mask as a
high-contrast cyan outline, so where the two agree and disagree is easy to spot
(see the legend below the figure).

In [ ]:
import ants
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# Pick a cross-validation case (skip the summary rows).
patient_id = results.loc[~results["id"].isin(summary_ids), "id"].iloc[0]

train_dir = Path("heart_mist/raw/train")
mri       = ants.image_read(str(train_dir / patient_id / "MRI.nii.gz")).numpy()
gt        = ants.image_read(str(train_dir / patient_id / "mask.nii.gz")).numpy()
raw_pred  = ants.image_read(f"heart_results/predictions/train/raw/{patient_id}.nii.gz").numpy()
post_pred = ants.image_read(f"heart_postprocessed/predictions/{patient_id}.nii.gz").numpy()

# Three axial slices around the middle of the predicted structure.
occupied = np.where(raw_pred.sum(axis=(0, 1)) > 0)[0]
mid = len(occupied) // 2
idx = np.clip(np.linspace(mid - 5, mid + 5, 3), 0, len(occupied) - 1).astype(int)
slices = occupied[idx]

pred_color = "#ff2d2d"   # red   — model prediction (filled)
gt_color   = "#00e5ff"   # cyan  — ground truth (outline)
pred_cmap  = ListedColormap([pred_color])

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
fig.suptitle(f"Patient: {patient_id}", fontsize=13)

for col, s in enumerate(slices):
    for row, (pred, title) in enumerate(
        [(raw_pred, "Raw prediction"), (post_pred, "Postprocessed")]
    ):
        ax = axes[row, col]
        ax.imshow(mri[:, :, s].T, cmap="gray", origin="lower")
        pred_mask = np.ma.masked_where(pred[:, :, s] == 0, pred[:, :, s])
        ax.imshow(pred_mask.T, cmap=pred_cmap, alpha=0.45, origin="lower")
        ax.contour(gt[:, :, s].T, levels=[0.5], colors=gt_color, linewidths=1.8)
        ax.set_title(f"{title}\nslice {s}")
        ax.axis("off")

legend_handles = [
    Patch(facecolor=pred_color, alpha=0.45, label="Prediction"),
    Line2D([0], [0], color=gt_color, lw=2, label="Ground truth"),
]
fig.legend(
    handles=legend_handles, loc="lower center", ncol=2,
    frameon=False, fontsize=11, bbox_to_anchor=(0.5, -0.01),
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()

## Section 10: What's Next

You've run the full MIST pipeline end to end — from raw NIfTI files to trained
models, evaluated predictions, postprocessing, ranking, and visualization. A few
directions from here:

**Go deeper.** This demo trained just a few epochs per fold with a small patch
size to run quickly. For real results, raise `--epochs`, drop `--patch-size` to let
MIST pick the optimal size, and explore other architectures (`--model`) and loss
functions (`--loss`). The [MIST documentation](https://mist-medical.readthedocs.io/)
covers every stage in detail.

**MIST is built to work with LLMs.** Several features are designed to hand off to
a model like Claude:

- `mist_analyze --data-dump` writes a `data_dump.md` summarizing spacing,
  intensity, and shape statistics — along with auto-generated observations — that
  you can paste into an LLM to get architecture, loss, and hyperparameter
  recommendations tailored to your dataset.
- `describe_transforms()` returns structured metadata for every postprocessing
  transform, the intended interface for LLM-proposed postprocessing strategies.
- The manual raw-vs-postprocessed comparison we did in Section 8 can be
  automated: [`mist-autoresearch`](https://github.com/mist-medical/mist-autoresearch)
  runs an autonomous LLM-in-the-loop that iteratively proposes and evaluates
  postprocessing strategies until it stops improving. *(Note: `mist-autoresearch`
  is still in active development.)*

**Use the `mist_expert` skill in Claude Code.** If you work on your own MIST
projects in [Claude Code](https://www.anthropic.com/claude-code), the
[`mist-medical/skills`](https://github.com/mist-medical/skills) repo ships a
`mist_expert` skill that gives Claude deep, structured knowledge of MIST — CLI
flags, `config.json` keys, and extension points — so it can help you configure,
debug, and extend pipelines. Install it by copying the skill into your project's
commands directory:

```bash
cp mist_expert.md /path/to/your/project/.claude/commands/mist_expert.md
```

then invoke `/mist_expert` in Claude Code. (This runs in Claude Code — the CLI,
desktop, or IDE extension — not inside this Colab notebook.)